In [1]:
with open(f"data/dialogues_train.txt", "r", encoding="utf-8") as f:
    text = f.read()

text = text.replace("__eou__ ", "")

chars = sorted(list(set(text)))
vocab_size = len(chars)

s_i = { ch : i for i, ch in enumerate(chars) }
i_s = { i : ch for i, ch in enumerate(chars) }

encode = lambda s: [s_i[c] for c in s]
decode = lambda l: "".join([i_s[i] for i in l])

In [33]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.8*len(data))

dtr = data[:n]
dval = data[n:]

torch.manual_seed(1337)
batch_size = 32
block_size = 8 # max context length

def get_batch(split):
    data = dtr if split == "train" else dval
    ix = torch.randint(len(data) - block_size, (batch_size,))
    
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i+1 : i+block_size+1] for i in ix])
    
    return x, y

In [45]:
from torch import nn
from torch.nn import functional as F

torch.manual_seed(1337)

n_embd = 32
lr = 1e-3
eval_iters = 500
eval_interval = 500
max_iters = 5000

n_heads = 4

In [46]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        
        self.key = nn.Linear(n_embd, head_size, bias=False) 
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
    
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        
        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        
        v = self.value(x)
        out = wei @ v
        
        return out

In [1]:
class Bigram(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embd = nn.Embedding(vocab_size, n_embd)
        self.position_embd = nn.Embedding(block_size, n_embd)
        
        self.sa_heads = MultiHeadAttention(n_heads, n_embd//n_heads)
        
        self.lm_head = nn.Linear(n_embd, vocab_size)
        
    
    def forward(self, idx, targets=None):
        B, T = idx.shape
        
        tok_emb = self.token_embd(idx) # (B, T, C)
        pos_emb = self.position_embd(torch.arange(T)) # (T, C)
        
        x = tok_emb + pos_emb # (B, T, C)
        x = self.sa_heads(x)
        
        logits = self.lm_head(x) # (B, T, vocab_size)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            
            targets = targets.view(-1)
            
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond) 
            logits = logits[:, -1, :] # (B, C)
            
            probs = F.softmax(logits, dim=-1) # (B,C)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)

        return idx

NameError: name 'nn' is not defined

In [65]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList((Head(head_size) for _ in range(num_heads)))
        
    def forward(self, x):
        return torch.cat([h(x) for h in self.heads], dim=-1)

In [66]:
net = Bigram()
optimizer = torch.optim.AdamW(net.parameters(), lr=1e-3)

In [67]:
context = torch.zeros((1, 1), dtype=torch.long)
print(decode(net.generate(context, max_new_tokens=100)[0].tolist()))


£、%bInj。~~Is/–u%!DIfKl3@#j“1‘”:)0OH%F\RR3$o\C。。gC)fNSr@1Qt=6°k3%;85dXjJVaJ—*(5°9!voVI‘K6QLdJrD8:4‘


In [68]:
@torch.no_grad()
def estimate_loss():
    out = {}
    net.eval()
    
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        
        for k in range(eval_iters):
            X, y = get_batch(split)
            logits, loss = net(X, y)
            losses[k] = loss.item()
            
        out[split] = losses.mean()
    
    net.train()
    return out

In [69]:
estimate_loss()

{'train': tensor(4.6591), 'val': tensor(4.6594)}

In [70]:
for step in range(max_iter):
    Xb, yb = get_batch("train")
    
    logits, loss = net(Xb, yb)
    
    optimizer.zero_grad(set_to_none=True)
    
    loss.backward()
    optimizer.step()
    
    if step % 500 == 0:
        print(f'{step} / {max_iter}: {loss.item()}')
        
print("Final loss:", loss)

0 / 10000: 4.639276027679443
500 / 10000: 2.556018590927124
1000 / 10000: 2.351381301879883
1500 / 10000: 2.273411512374878
2000 / 10000: 2.2011632919311523
2500 / 10000: 2.1930930614471436
3000 / 10000: 2.187005043029785
3500 / 10000: 2.1994130611419678
4000 / 10000: 2.034649133682251
4500 / 10000: 2.1149632930755615
5000 / 10000: 2.182417631149292
5500 / 10000: 2.025371789932251
6000 / 10000: 2.238621950149536
6500 / 10000: 2.0466456413269043
7000 / 10000: 2.0406174659729004
7500 / 10000: 1.9473167657852173
8000 / 10000: 2.0733296871185303
8500 / 10000: 2.1421010494232178
9000 / 10000: 2.0257034301757812
9500 / 10000: 1.9708635807037354
Final loss: tensor(1.9319, grad_fn=<NllLossBackward0>)
